# 统一过滤 Subclass 后重跑十组实验

以每组原真实训练集中每个 Subclass 的细胞数为准，保留至少 10 个细胞的类别。同一掩码规则用于真实训练集和留出测试集。沿用原细胞身份、区域轴及 1,000 对复制区域。

In [ ]:
from pathlib import Path
import json
import shutil
import anndata as ad
from data.paths import OUTPUTS, check_output, require_environment
from data.preprocessing import filter_training_subclasses
from data.evaluation import region_pairs, validate_real_duplicates

CASES = [
    'liu2023/full', 'liu2023/GSE213221', 'liu2023/GSE215213', 'liu2023/GSE215342',
    'liu2021/GSE130553', 'liu2021/GSE131354', 'liu2021/GSE131360',
    'tian2023/GSE167119', 'tian2023/GSE167577', 'tian2023/GSE167978',
]
ROOT = check_output(OUTPUTS / 'min10')

def prepare_filtered_case(case):
    original = OUTPUTS / 'preprocessing' / case
    output = ROOT / 'preprocessing' / case
    report_path = output / 'filter.json'
    if report_path.is_file():
        return json.loads(report_path.read_text(encoding='utf-8'))
    counts = filter_training_subclasses(
        original / 'TrainingReal.h5ad', original / 'HeldoutReal.h5ad', output, min_cells=10
    )
    pairs = original / 'region_pairs.csv'
    for name in ('TrainingReal', 'HeldoutReal'):
        item = ad.read_h5ad(output / f'{name}.h5ad', backed='r')
        try:
            validate_real_duplicates(item, region_pairs(item, pair_path=pairs))
        finally:
            item.file.close()
    shutil.copyfile(pairs, output / pairs.name)
    counts.to_csv(output / 'Subclass细胞数.csv', encoding='utf-8-sig')
    totals = counts[['训练细胞数', '测试细胞数']].sum()
    kept = counts.loc[counts['保留'], ['训练细胞数', '测试细胞数']].sum()
    report = {
        '数据集': case, '训练最少细胞数': 10, '过滤前': totals.to_dict(), '过滤后': kept.to_dict(),
        '保留Subclass数': int(counts['保留'].sum()), '过滤Subclass数': int((~counts['保留']).sum()),
    }
    temporary = report_path.with_suffix('.json.partial')
    temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(report_path)
    return report

In [ ]:
require_environment()
reports = [prepare_filtered_case(case) for case in CASES]
(ROOT / 'Subclass过滤汇总.json').write_text(
    json.dumps(reports, ensure_ascii=False, indent=2), encoding='utf-8'
)
for report in reports:
    print(report, flush=True)